# Hybrid2
Implémentation de la méthode décrite dans :
> *Hybrid Approach To Unsupervised Keyphrase Extraction*  
> Singh & Bolla, Procedia Computer Science 235 (2024) 1498–1511

## Pipeline

```
Texte brut
    ↓  clean_text()          — lowercase + suppression caractères spéciaux
Texte nettoyé
    ↓        ↓        ↓
 TopicRank  YAKE    RAKE    — extraction indépendante
    ↓        ↓        ↓
   Normalisation min-max    — rend les scores comparables entre méthodes
         ↓
    Fusion Hybrid2           — somme des scores normalisés par keyphrase
         ↓
  Re-normalisation [0,1]    — score final uniforme
```

## Choix clés conformes au papier

| Étape | Décision | Justification (papier) |
|-------|----------|------------------------|
| Pré-traitement | Texte nettoyé uniquement | Table 5 : F1=0 avec stemming |
| Normalisation | Min-max indépendante par méthode | TopicRank ~0.03, RAKE ~15 : scales incompatibles |
| N-grammes | Max trigrammes | Section 3.2.1 : les n-grammes longs dégradent les résultats |
| Score final | Re-normalisé en [0,1] | Ajout pratique pour comparaison directe |

## 1. Installation des dépendances

Les bibliothèques nécessaires sont :
- **pke** : implémentation de TopicRank
- **yake** : implémentation officielle de YAKE
- **rake-nltk** : implémentation de RAKE
- **spaCy** : modèles de langue pour TopicRank (tokenisation, POS tagging)
- **nltk** : stopwords pour RAKE

In [ ]:
!pip install yake rake-nltk nltk --quiet
!pip install git+https://github.com/boudinfl/pke.git --quiet
!python -m spacy download en_core_web_sm --quiet
!python -m spacy download fr_core_news_sm --quiet
!python -m nltk.downloader stopwords punkt punkt_tab --quiet

## 2. Imports

In [ ]:
import re
import os
import time
from typing import List, Tuple, Dict
import yake
import pke
import nltk
from rake_nltk import Rake
from nltk.corpus import stopwords as nltk_stopwords

## 3. Paramètres

Modifier `LANGUE` et `N_KEYWORDS` selon les besoins.

| Paramètre | Valeurs possibles | Description |
|-----------|-------------------|-------------|
| `LANGUE` | `'en'`, `'fr'` | Langue du corpus |
| `N_KEYWORDS` | entier positif | Nombre de keyphrases à extraire |

In [ ]:
LANGUE     = 'en'   # 'en' or 'fr'
N_KEYWORDS = 15     # number of keyphrases to extract
print(f'Langue : {LANGUE} | N keyphrases : {N_KEYWORDS}')

## 4. Corpus

Un corpus d'exemple est fourni ci-dessous. Trois options sont disponibles pour utiliser vos propres documents :

- **Option A** : saisir le texte directement dans la variable `CORPUS`
- **Option B** : charger des fichiers `.txt` depuis le PC local
- **Option C** : lire des fichiers depuis Google Drive

In [ ]:
# --- Example corpus (replace with your own documents) ---
CORPUS = [
    """Computer programming or coding is the composition of sequences of instructions,
    called programs, that computers can follow to perform tasks. It involves designing
    and implementing algorithms, step-by-step specifications of procedures, by writing
    code in one or more programming languages. Programmers typically use high-level
    programming languages that are more easily intelligible to humans than machine code,
    which is directly executed by the central processing unit.""",

    """Software development is the process of conceiving, specifying, designing,
    programming, documenting, testing, and bug fixing involved in creating and maintaining
    applications, frameworks, or other software components. Software development involves
    writing and maintaining the source code, but in a broader sense, it includes all
    processes from the conception of the desired software through to the final manifestation
    of the software, typically in a planned and structured process.""",

    """An algorithm is a finite sequence of rigorous instructions, typically used to solve
    a class of specific problems or to perform a computation. Algorithms are used as
    specifications for performing calculations and data processing. More advanced algorithms
    can use conditionals to divert the code execution through various routes, referred to
    as automated decision-making. Algorithms are at the core of every computer program.""",

    """A programming language is a system of notation for writing computer programs.
    Programming languages are described in terms of their syntax and semantics. Most
    programming languages consist of instructions for computers. There are thousands of
    programming languages and new ones are created every year. A few hundred programming
    languages are commonly used. Programming languages differ from natural languages.""",

    """Object-oriented programming is a programming paradigm based on the concept of objects,
    which can contain data and code. The data is in the form of fields, often known as
    attributes or properties, and the code is in the form of procedures, often known as
    methods. A common feature of objects is that methods are attached to them and can
    access and modify the object's data fields.""",
]

print(f'{len(CORPUS)} documents loaded')

In [ ]:
# --- Option B: load .txt files from local machine ---
from google.colab import files
uploaded = files.upload()
CORPUS = [v.decode('utf-8') for v in uploaded.values()]

# --- Option C: load .txt files from Google Drive ---
# from google.colab import drive
# import glob
# drive.mount('/content/drive')
# CORPUS = [open(f).read() for f in glob.glob('/content/drive/MyDrive/corpus/*.txt')]

print('Décommenter une option ci-dessus pour charger votre propre corpus.')

## 5. Pré-traitement

Conformément à la **section 3.2 du papier**, le pré-traitement se limite au texte nettoyé (*cleaned text*) :

1. Conversion en minuscules
2. Suppression du balisage wiki (`'''`, `[[]]`, `{{}}`, etc.)
3. Suppression des caractères non alphabétiques
4. Normalisation des espaces

> **Le stemming est volontairement exclu** : la Table 5 du papier montre que le stemming donne un F1=0 sur tous les modèles testés.

Le même texte nettoyé est transmis aux trois méthodes. Chaque méthode gère ensuite **en interne** la tokenisation, la suppression des stopwords et la sélection des candidats.

In [ ]:
def clean_text(text: str) -> str:
    """
    Clean raw text following paper section 3.2:
    lowercase → remove wiki markup → remove non-alphabetic chars → normalize spaces.
    Stemming is explicitly excluded (paper Table 5: F1=0 with stemming).
    """
    text = text.lower()
    # Remove wiki markup: '''bold''', ''italic'', [[link|label]], {{template}}
    text = re.sub(r"'{2,3}", ' ', text)
    text = re.sub(r'\[\[([^|\]]+\|)?([^\]]+)\]\]', r'\2', text)
    text = re.sub(r'\{\{[^}]*\}\}', ' ', text)
    text = re.sub(r'\[https?://\S+\s*([^\]]*)\]', r'\1', text)
    # Keep only alphabetic characters and spaces (including accented letters)
    text = re.sub(r'[^a-zàâäéèêëîïôùûüç\s]', ' ', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Apply unified pre-processing to the entire corpus
cleaned_corpus = [clean_text(doc) for doc in CORPUS]

# Preview
print('=== Raw text (first 200 chars) ===')
print(CORPUS[0][:200])
print('\n=== Cleaned text (first 200 chars) ===')
print(cleaned_corpus[0][:200])

## 6. Normalisation min-max

Les trois méthodes produisent des scores sur des **échelles très différentes** :

| Méthode | Échelle typique | Sens |
|---------|-----------------|------|
| TopicRank | ~0.001 – 0.05 | Plus élevé = meilleur |
| YAKE | ~0.0001 – 0.02 | **Plus bas = meilleur** |
| RAKE | ~5 – 20 | Plus élevé = meilleur |

Sans normalisation, RAKE dominerait complètement la fusion (ses scores sont ~1000× plus grands). La normalisation min-max ramène chaque méthode dans `[0, 1]` **indépendamment**, puis inverse les scores YAKE (`1 - norm`) pour que dans tous les cas un score élevé signifie une meilleure keyphrase.

In [ ]:
# Apply min-max normalisation to a list of (keyphrase, score) tuples
# For YAKE (lower score = better keyphrase), scores are inverted after normalisation so that higher always means more relevant
# Args:
#   results          : list of (keyphrase, raw_score)
#   higher_is_better : set to False for YAKE
# Returns:
#   dict {keyphrase_lowercase: normalised_score in [0, 1]}
def min_max_normalise(
    results: List[Tuple[str, float]],
    higher_is_better: bool = True
) -> Dict[str, float]:
    if not results:
        return {}
    scores = [s for _, s in results]
    min_s, max_s = min(scores), max(scores)
    denom = max_s - min_s if max_s != min_s else 1.0
    out = {}
    for kp, score in results:
        norm = (score - min_s) / denom
        out[kp.lower().strip()] = (1.0 - norm) if not higher_is_better else norm
    return out

print('min_max_normalise() définie')

## 7. Méthode 1 — TopicRank

TopicRank est une méthode **basée sur les graphes** :

1. Extraction des candidats (groupes nominaux via POS tagging spaCy)
2. Regroupement des candidats similaires en *topics* (similarité de Jaccard)
3. Construction d'un graphe de topics pondéré par les co-occurrences
4. Score par PageRank — le meilleur candidat de chaque topic est sélectionné

La bibliothèque `pke` gère en interne tokenisation, stopwords et sélection des candidats.

In [ ]:
# Run TopicRank via the pke library
# Input : cleaned text from clean_text()
# Score : higher is better (PageRank-based)
# pke requires a loaded spaCy model object — loading it outside the function and passing it as argument avoids reloading from disk on every call
# Tokenisation, stopword removal and candidate selection are handled internally by pke/spaCy
# Args:
#   cleaned_corpus : list of cleaned document strings
#   lang           : 'en' or 'fr'
#   n              : number of keyphrases to return
#   nlp            : pre-loaded spaCy model (loaded once outside to save time)
# Returns:
#   (list of (keyphrase, score), elapsed_time_seconds)
def run_topicrank(
    cleaned_corpus : List[str],
    lang           : str,
    n              : int,
    nlp            = None   # pre-loaded spaCy model — pass it to avoid reloading on every call
) -> Tuple[List[Tuple[str, float]], float]:
    import spacy
    full_text  = ' '.join(cleaned_corpus)

    # Load the spaCy model only if not provided
    if nlp is None:
        spacy_model_name = 'fr_core_news_sm' if lang == 'fr' else 'en_core_web_sm'
        nlp = spacy.load(spacy_model_name)

    t0 = time.time()
    extractor = pke.unsupervised.TopicRank()
    extractor.load_document(input=full_text, language=lang, spacy_model=nlp)
    extractor.candidate_selection()
    extractor.candidate_weighting()
    results = extractor.get_n_best(n=n)
    elapsed = time.time() - t0

    return results, elapsed


# Load the spaCy model once here — reused across all calls to run_topicrank()
import spacy
spacy_model_name = 'fr_core_news_sm' if LANGUE == 'fr' else 'en_core_web_sm'
nlp = spacy.load(spacy_model_name)
print(f'Modèle spaCy chargé : {spacy_model_name}')

tr_results, tr_time = run_topicrank(cleaned_corpus, LANGUE, N_KEYWORDS * 2, nlp=nlp)
print(f'TopicRank terminé en {tr_time:.2f}s — {len(tr_results)} keyphrases extraites')

## 8. Méthode 2 — YAKE

YAKE (*Yet Another Keyword Extractor*, Campos et al., 2018) est une méthode **statistique** qui combine des features lexicales : casse, position dans la phrase, fréquence et dispersion.

> **Convention inversée** : pour YAKE, un score **plus bas** indique une keyphrase **plus pertinente**. La normalisation au step suivant gère cette inversion automatiquement (`higher_is_better=False`).

In [ ]:
# Run YAKE via the official yake library
# Input : cleaned text from clean_text()
# Score : LOWER is better — inversion is handled in min_max_normalise()
# Tokenisation, stopword removal and candidate selection are handled internally
# Args:
#   cleaned_corpus : list of cleaned document strings
#   lang           : 'en' or 'fr'
#   n              : number of keyphrases to return
# Returns:
#   (list of (keyphrase, score), elapsed_time_seconds)
def run_yake(
    cleaned_corpus: List[str],
    lang: str,
    n: int
) -> Tuple[List[Tuple[str, float]], float]:
    full_text  = ' '.join(cleaned_corpus)
    yake_lang  = 'fr' if lang == 'fr' else 'en'

    t0 = time.time()
    extractor = yake.KeywordExtractor(
        lan=yake_lang,
        n=3,         # max n-gram size: unigrams, bigrams, trigrams (paper section 3.2.1)
        dedupLim=0.7,
        top=n
    )
    results = extractor.extract_keywords(full_text)
    elapsed = time.time() - t0

    return results, elapsed


yake_results, yake_time = run_yake(cleaned_corpus, LANGUE, N_KEYWORDS * 2)
print(f'YAKE terminé en {yake_time:.2f}s — {len(yake_results)} keyphrases extraites')

## 9. Méthode 3 — RAKE

RAKE (*Rapid Automatic Keyword Extraction*, Rose et al., 2010) est une méthode **statistique** basée sur le ratio degré/fréquence des mots.

Le texte est découpé en candidats en utilisant les stopwords comme délimiteurs. Le score d'un candidat est la somme des scores `deg(w)/freq(w)` de ses tokens.

Chaque document est traité **séparément** pour éviter qu'un corpus pré-traité (sans ponctuation) produise un unique candidat géant. La bibliothèque `rake-nltk` gère en interne la suppression des stopwords.

In [ ]:
# Run RAKE via the rake-nltk library
# Input : cleaned text from clean_text()
# Each document is processed separately to avoid one giant candidate when the corpus has no punctuation
# Score : higher is better
# Stopword removal and candidate selection are handled internally by rake-nltk
# Args:
#   cleaned_corpus : list of cleaned document strings
#   lang           : 'en' or 'fr'
#   n              : number of keyphrases to return
# Returns:
#   (list of (keyphrase, score), elapsed_time_seconds)
def run_rake(
    cleaned_corpus: List[str],
    lang: str,
    n: int
) -> Tuple[List[Tuple[str, float]], float]:
    nltk_lang = 'french' if lang == 'fr' else 'english'
    try:
        sw = nltk_stopwords.words(nltk_lang)
    except LookupError:
        nltk.download('stopwords', quiet=True)
        sw = nltk_stopwords.words(nltk_lang)

    t0 = time.time()
    cand_scores: Dict[str, float] = {}

    for doc in cleaned_corpus:
        r = Rake(
            stopwords=sw,
            min_length=1,
            max_length=3   # max trigrams, following paper section 3.2.1
        )
        r.extract_keywords_from_text(doc)
        for score, kp in r.get_ranked_phrases_with_scores():
            kp_lower = kp.lower().strip()
            # Keep the highest score if the same candidate appears in multiple documents
            if kp_lower not in cand_scores or score > cand_scores[kp_lower]:
                cand_scores[kp_lower] = score

    elapsed = time.time() - t0
    results = sorted(cand_scores.items(), key=lambda x: x[1], reverse=True)[:n]
    return results, elapsed


rake_results, rake_time = run_rake(cleaned_corpus, LANGUE, N_KEYWORDS * 2)
print(f'RAKE terminé en {rake_time:.2f}s — {len(rake_results)} keyphrases extraites')

## 10. Méthode 4 — Hybrid2

Implémentation de l'**Algorithm 2** du papier (section 3.2.2.2), avec re-normalisation finale ajoutée pour obtenir un score dans `[0, 1]`.

```
Algorithm 2 — Hybrid2
─────────────────────────────────────────────────────
1. Input  : résultats de TopicRank, YAKE, RAKE + N
2. Normaliser les scores de chaque modèle (min-max)
3. Créer un dictionnaire {keyphrase: score_cumulé}
4. Pour chaque keyphrase dans les 3 modèles :
     - Si présente → ajouter le score au dictionnaire
     - Si nouvelle  → l'insérer avec son score
   Trier par score décroissant
   S'arrêter quand N keyphrases atteintes
5. Output : liste finale
─────────────────────────────────────────────────────
```

**Pourquoi la normalisation est essentielle** : une keyphrase trouvée uniquement par RAKE aurait un avantage injuste sans normalisation (score brut ~15 contre ~0.03 pour TopicRank). Après normalisation, chaque méthode contribue au maximum avec un score de 1.

In [ ]:
# Hybrid2 fusion following Algorithm 2 of the paper (section 3.2.2.2)
# Steps:
#   1. Min-max normalise each method independently (YAKE inverted: lower=better → higher_is_better=False)
#   2. Fusion dict: for each keyphrase, sum normalised scores across all methods that contain it
#   3. Sort by descending fused score
#   4. Select top-n keyphrases
#   5. Final re-normalisation to [0, 1] for uniform output scores
# Args:
#   topicrank_results : list of (keyphrase, score) from TopicRank
#   yake_results      : list of (keyphrase, score) from YAKE
#   rake_results      : list of (keyphrase, score) from RAKE
#   n                 : number of keyphrases to return
# Returns:
#   (list of (keyphrase, score in [0,1]), elapsed_time_seconds)
def run_hybrid2(
    topicrank_results : List[Tuple[str, float]],
    yake_results      : List[Tuple[str, float]],
    rake_results      : List[Tuple[str, float]],
    n                 : int
) -> Tuple[List[Tuple[str, float]], float]:
    t0 = time.time()

    # Step 1: independent min-max normalisation per method
    norm_tr   = min_max_normalise(topicrank_results, higher_is_better=True)
    norm_yake = min_max_normalise(yake_results,      higher_is_better=False)  # YAKE: lower = better
    norm_rake = min_max_normalise(rake_results,      higher_is_better=True)

    # Step 2: fusion by score summation (Algorithm 2, lines 3-4)
    fused: Dict[str, float] = {}
    for method_dict in (norm_tr, norm_yake, norm_rake):
        for kp, score in method_dict.items():
            fused[kp] = fused.get(kp, 0.0) + score

    # Step 3-4: sort descending, select top-n (Algorithm 2, lines 4-5)
    sorted_kp = sorted(fused.items(), key=lambda x: x[1], reverse=True)
    top_n = sorted_kp[:n]

    if not top_n:
        return [], time.time() - t0

    # Step 5: final re-normalisation to [0, 1]
    top_scores = [s for _, s in top_n]
    mn, mx = min(top_scores), max(top_scores)
    denom = mx - mn if mx != mn else 1.0
    final = [(kp, (s - mn) / denom) for kp, s in top_n]

    return final, time.time() - t0


h2_results, h2_time = run_hybrid2(tr_results, yake_results, rake_results, N_KEYWORDS)
print(f'Hybrid2 terminé en {h2_time:.2f}s — {len(h2_results)} keyphrases extraites')

## 11. Affichage des résultats

Fonction utilitaire pour afficher les résultats sous forme de tableau formaté.

In [ ]:
# Print a formatted ranking table with keyphrase and score columns
def print_table(
    title   : str,
    results : List[Tuple[str, float]],
    elapsed : float,
    note    : str = ''
) -> None:
    if not results:
        print(f'\n[{title}] Aucun résultat.')
        return
    w   = max(max(len(kp) for kp, _ in results), 20)
    sep = '-' * (w + 30)
    hdr = f'  {title}' + (f'  ({note})' if note else '')
    print(f'\n+{sep}+')
    print(f'|{hdr:<{len(sep)}}|')
    print(f'|  Temps : {elapsed:.2f}s{"":<{len(sep) - 15}}|')
    print(f'+{sep}+')
    print(f'|  {"#":<5}{"Keyphrase":<{w}}  {"Score":>8}  |')
    print(f'+{sep}+')
    for i, (kp, sc) in enumerate(results, 1):
        print(f'|  {i:<5}{kp:<{w}}  {sc:>8.4f}  |')
    print(f'+{sep}+')

# Print keyphrase overlap between the four methods
def print_overlap(
    tr   : List[Tuple[str, float]],
    yake : List[Tuple[str, float]],
    rake : List[Tuple[str, float]],
    h2   : List[Tuple[str, float]],
    n    : int
) -> None:
    set_tr   = {kp.lower() for kp, _ in tr[:n]}
    set_yake = {kp.lower() for kp, _ in yake[:n]}
    set_rake = {kp.lower() for kp, _ in rake[:n]}
    set_h2   = {kp.lower() for kp, _ in h2}
    pairs = [
        ('TopicRank & YAKE',       set_tr   & set_yake),
        ('TopicRank & RAKE',       set_tr   & set_rake),
        ('YAKE      & RAKE',       set_yake & set_rake),
        ('Tous les 3',             set_tr   & set_yake & set_rake),
        ('Hybrid2   & TopicRank',  set_h2   & set_tr),
        ('Hybrid2   & YAKE',       set_h2   & set_yake),
        ('Hybrid2   & RAKE',       set_h2   & set_rake),
    ]
    sep = '-' * 72
    print(f'\n{sep}')
    print('  CHEVAUCHEMENT ENTRE MÉTHODES')
    print(sep)
    print(f'  {"Paire":<28}  {"N":>3}  Keyphrases communes')
    print(sep)
    for label, inter in pairs:
        kps = ', '.join(sorted(inter)) if inter else '-'
        if len(kps) > 36:
            kps = kps[:33] + '...'
        print(f'  {label:<28}  {len(inter):>3}  {kps}')
    print(sep)

print('Fonctions d\'affichage définies')

## 12. Résultats

Affichage des tableaux de résultats pour les quatre méthodes, suivi d'une analyse du chevauchement entre les keyphrases extraites.

In [ ]:
print_table('TOPICRANK', tr_results[:N_KEYWORDS],   tr_time,   note='higher = better')
print_table('YAKE',      yake_results[:N_KEYWORDS], yake_time, note='lower  = better')
print_table('RAKE',      rake_results[:N_KEYWORDS], rake_time, note='higher = better')
print_table('HYBRID2',   h2_results,                h2_time,   note='scores fusionnés normalisés [0,1]')
print_overlap(tr_results, yake_results, rake_results, h2_results, N_KEYWORDS)

## 13. Avantages et inconvénients

**Avantages**
- Très léger computationnellement : aucun modèle neuronal, aucun GPU nécessaire. Fonctionne sur n'importe quelle machine.
- Multilingue nativement : YAKE et RAKE supportent `en` et `fr` out-of-the-box, TopicRank via `pke` également.
- Couvre trois paradigmes complémentaires : graphe (TopicRank), statistique-positionnel (YAKE), statistique-structurel (RAKE) — la fusion peut compenser les angles morts de chaque méthode.
- Fidèle au papier (Singh & Bolla, 2024) : l'Algorithme 2 est implémenté tel que décrit, avec la normalisation min-max nécessaire pour rendre comparables les échelles hétérogènes des trois méthodes.

**Inconvénients**
- **Le score de fusion est une simple somme de scores normalisés** : Hybrid2 n'apprend aucun poids optimal pour combiner les méthodes — TopicRank, YAKE et RAKE contribuent toujours de façon équivalente, indépendamment du domaine ou de la qualité relative des méthodes sur ce corpus. Aucune supervision, aucun apprentissage.
- **Propagation des limites de chaque méthode** : TopicRank est sensible à la qualité du POS tagging (spaCy peut se tromper sur les textes très techniques) ; RAKE est fragile sur les textes sans ponctuation (comme les abstracts après `clean_text()`, qui supprime les caractères spéciaux) ; YAKE pénalise les termes apparaissant dans de nombreux contextes différents, ce qui peut désavantager des termes généraux mais pertinents au domaine.
- **Aucune représentation sémantique** : les trois méthodes sont purement lexicales et statistiques. Deux keyphrases synonymes sont traitées comme des entités totalement distinctes — aucune capacité à capturer la similarité sémantique qu'ELMo ou SBERT capturent.
- La normalisation min-max est sensible aux valeurs aberrantes : un seul candidat avec un score très élevé peut comprimer tous les autres vers zéro, dégradant le classement final.
- `clean_text()` supprime toute la ponctuation avant de passer le texte à RAKE, ce qui prive RAKE de ses délimiteurs naturels pour la construction des candidats — RAKE est donc appliqué sur un texte déjà dégradé par rapport à son entrée idéale.